# S.T.A.R. Sky Projection Pipeline
**Symbolic–Topological–Arithmetic → Cosmic Projection + Real Galaxy Overlay**

In [ ]:
# 1. Imports & Project Root Setup
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def get_project_root():
    cwd = Path.cwd().resolve()
    for parent in [cwd] + list(cwd.parents):
        if (parent / "src").exists():
            return parent
    return cwd

ROOT = get_project_root()
sys.path.insert(0, str(ROOT))
print(" Project root:", ROOT)


In [ ]:
# 2. Load core S.T.A.R. modules
from src.acsc.projection import ArithmeticProjector
from src.tda.persistence import compute_persistence

print(" ACSC and TDA modules loaded.")

In [ ]:
# 3. Load Real Galaxy Data for Overlay
galaxy_paths = [
    ROOT / "data" / "cosmic" / "DESIDR8_SDSSDR16_SIMBAD.csv",
    ROOT / "data" / "cosmic" / "galaxy_sample.csv",
    # Add more paths if you have other catalogs
]

galaxies = None
for path in galaxy_paths:
    if path.exists():
        galaxies = pd.read_csv(path)
        print(f" Loaded {len(galaxies)} real galaxies from {path.name}")
        break

if galaxies is None:
    print(" No galaxy catalog found. Generating synthetic galaxies.")
    np.random.seed(42)
    galaxies = pd.DataFrame({
        'ra': np.random.uniform(0, 360, 5000),
        'dec': np.random.uniform(-90, 90, 5000),
        'z': np.random.lognormal(0, 0.6, 5000).clip(0.01, 2.5),
        'logM': np.random.normal(10.5, 0.8, 5000)
    })

In [ ]:
# 4. Generate Arithmetic Cloud & Project
np.random.seed(42)
n_samples = 8000

arithmetic_data = {
    'delta': np.random.lognormal(0, 1.8, n_samples),
    'conductor': np.random.lognormal(4, 2.2, n_samples),
    'rank': np.random.randint(0, 5, n_samples),
    'regulator': np.random.lognormal(0, 1.4, n_samples)
}

projector = ArithmeticProjector()

print("Projecting arithmetic invariants to cosmic manifold...")
arithmetic_cloud = projector.project(arithmetic_data)
embedded = projector.embed_to_cosmic(arithmetic_cloud)

print(f" Projection completed. Embedded shape: {embedded.shape}")

In [ ]:
# 5. Advanced 3D Visualization
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Arithmetic Projection
scatter_arith = ax.scatter(embedded[:, 0], embedded[:, 1], embedded[:, 2],
                           c=arithmetic_data['rank'], cmap='viridis',
                           s=8, alpha=0.7, label='Arithmetic Projection')

# Real Galaxies Overlay
if len(galaxies) > 0:
    gal_subset = galaxies.sample(min(3000, len(galaxies)))
    ax.scatter(gal_subset['ra']/12, gal_subset['dec']/6, gal_subset['z']*10,
               c='red', s=3, alpha=0.35, label='Real Galaxies')

ax.set_xlabel('Cosmic X')
ax.set_ylabel('Cosmic Y')
ax.set_zlabel('Cosmic Z')
ax.set_title('S.T.A.R. Arithmetic → Cosmic Projection\nwith Real Galaxy Overlay')
plt.legend()
plt.show()

In [ ]:
# 6. Topological Analysis
print("Computing persistence diagrams...")
arith_pd = compute_persistence(arithmetic_cloud, max_dim=2)
cosmic_pd = compute_persistence(embedded, max_dim=2)
print(" Persistence diagrams computed.")

# 7. Integration Note
print("\nReady for integration with PaperFiguresPipeline and JointMCMCPipeline.")

## Results & Next Steps

Arithmetic invariants successfully projected into cosmic coordinates.
Real galaxy catalog overlaid for visual comparison.
Framework prepared for integration with PaperFiguresPipeline.

**Future Enhancements:**

Compute Wasserstein distance between arithmetic & galaxy persistence diagrams
Use projection output as prior in MCMC (JointMCMCPipeline)
Entropy field overlay (ECC)